In [ ]:
# Google Colab Only Version (No Streamlit)
# Run each section in Colab

!pip -q install plotly scikit-learn

import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go
from google.colab import files
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LinearRegression
from IPython.display import display, HTML

print('Upload your CSV file')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print('Dataset Loaded')
display(df.head())

def find_col(cols, keys):
    for c in cols:
        lc=c.lower()
        if any(k in lc for k in keys): return c
    return None

tcol=find_col(df.columns,['temp'])
hcol=find_col(df.columns,['humid','rh'])
assert tcol and hcol, 'Temperature/Humidity columns not found'

df=df[[tcol,hcol]].dropna().copy()
df.columns=['Temperature','Humidity']

def risk(r):
    if r['Temperature']>45 or r['Humidity']>80: return 'Critical'
    if r['Temperature']>35 or r['Humidity']>65: return 'Warning'
    return 'Safe'

df['Risk']=df.apply(risk,axis=1)
iso=IsolationForest(contamination=0.1, random_state=42)
df['Anomaly']=iso.fit_predict(df[['Temperature','Humidity']])
accuracy=(df['Anomaly']==1).mean()*100
print('AI Accuracy Score:', round(accuracy,2),'%')

if (df['Temperature']>45).any():
    display(HTML('<h2 style="color:red;">CRITICAL WARNING: Temperature exceeded 45°C</h2>'))

print(df.head())

counts=df['Risk'].value_counts().reset_index()
counts.columns=['Risk','Count']
fig=px.pie(counts, names='Risk', values='Count', title='Healthy vs Critical', color='Risk', color_discrete_map={'Safe':'green','Warning':'orange','Critical':'red'})
fig.update_traces(textinfo='percent+label+value', textfont_size=14, marker=dict(line=dict(color='black', width=2)))
fig.show()

from IPython.display import display, HTML
summary_html = '<div style="display:flex;gap:15px;">' + ''.join([f"<div style='border:2px solid black;padding:12px;background:white;width:180px;text-align:center;'><h3>{r}</h3><h2>{c}</h2></div>" for r,c in zip(counts['Risk'],counts['Count'])]) + '</div>'
display(HTML(summary_html))

fig=px.bar(counts, x='Risk', y='Count', title='Risk Counts', color='Risk', color_discrete_map={'Safe':'green','Warning':'orange','Critical':'red'}, text='Count')
fig.update_traces(marker_line_color='black', marker_line_width=2, textposition='outside')
fig.show()

df['Time']=range(len(df))
model=LinearRegression().fit(df[['Time']], df['Temperature'])
future=np.arange(len(df), len(df)+10).reshape(-1,1)
pred=model.predict(future)
fig=go.Figure()
fig.add_scatter(x=df['Time'], y=df['Temperature'], mode='lines', name='Historic')
fig.add_scatter(x=list(range(len(df),len(df)+10)), y=pred, mode='lines', name='Forecast')
fig.update_layout(title='Temperature Trend')
fig.show()

fig=px.scatter(df, x='Temperature', y='Humidity', color=df['Anomaly'].map({1:'Normal',-1:'Outlier'}), title='Anomaly Detection')
fig.show()

print('\nRecommendations:')
if 'Critical' in df['Risk'].values:
    print('- Check Ventilation')
    print('- Move Inventory')
    print('- Inspect Cooling System')
else:
    print('- Storage conditions are stable')


Upload your CSV file


Saving agri_storage.csv to agri_storage (1).csv
Dataset Loaded


,Storage_ID,Location,Produce_Type,Storage_Type,Capacity_Tons,Current_Stock_Tons,Temperature_C,Humidity_Percent,Last_Inspection_Date,Condition_Status
0,ST001,Karnataka,Wheat,Silo,500,320,18,55,2026-03-12,Good
1,ST002,Maharashtra,Rice,Warehouse,800,610,22,60,2026-02-25,Good
2,ST003,Punjab,Maize,Cold Storage,300,210,10,70,2026-03-05,Fair
3,ST004,Uttar Pradesh,Potato,Cold Storage,450,400,4,85,2026-03-18,Good
4,ST005,Tamil Nadu,Onion,Warehouse,350,290,25,65,2026-02-28,Fair


AI Accuracy Score: 90.0 %
   Temperature  Humidity      Risk  Anomaly
0           18        55      Safe        1
1           22        60      Safe        1
2           10        70   Warning        1
3            4        85  Critical       -1
4           25        65      Safe        1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning:

X does not have valid feature names, but LinearRegression was fitted with feature names




Recommendations:
- Check Ventilation
- Move Inventory
- Inspect Cooling System
